# VIDEO STUDIO AI — Producción cinematográfica Wan 2.2 + Supabase

Este es el notebook operativo que corresponde al repositorio privado y al worker de producción en `main`.

**Arquitectura:** Cloudflare Pages prepara el guion, manifiesto, fichas de personajes y una imagen maestra generada para cada segmento. Las imágenes se guardan en el bucket privado `video-assets`. Kaggle descarga esas imágenes exactas, anima cada una con Wan 2.2 I2V, encadena los fotogramas finales dentro de cada segmento, conserva los maestros en `video-renders`, genera el audio local con Piper/FFmpeg y sube el MP4 final a Supabase.

**Antes de ejecutar:** activa GPU NVIDIA e Internet en Kaggle. Configura en Kaggle Secrets los nombres exactos `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY` y `GITHUB_TOKEN`. Nunca pegues sus valores en las celdas.

**Orden:** ejecuta las celdas 1 a 6 en orden. La celda 5 valida una inferencia real I2V de aproximadamente 2 segundos. La celda 6 inicia el worker. Mantén activa la sesión GPU mientras haya trabajos.

**Importante:** cada segmento debe tener su imagen maestra generada y guardada en Estudio de producción cinematográfica antes de enviarse a Wan. La API rechaza trabajos incompletos. El trabajador conserva el progreso y las referencias. Kaggle gratuito no es un servidor permanente: si se desconecta, reanuda cuando vuelvas a ejecutar el notebook.

**Costos:** no se usa un proveedor de video de pago. Workers AI de Cloudflare dispone de una asignación gratuita diaria, pero el plan Workers Paid puede facturar el uso que supere esa asignación. La aplicación limita las llamadas a 10 imágenes y 10 voces por usuario/día; para asegurar cero cargos, conserva tu cuenta en Workers Free y no habilites un plan de pago. Kaggle GPU también está sujeto a cuota y disponibilidad.

In [ ]:
# CELDA 1 — Validar Kaggle, CUDA, espacio y rutas canónicas
import os, sys, pathlib, shutil, subprocess, platform

if sys.platform != "linux" or not pathlib.Path("/kaggle").is_dir():
    raise RuntimeError("Abre este notebook dentro de Kaggle, no en una vista previa.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No se detecta NVIDIA. En Kaggle selecciona Accelerator = GPU.")

subprocess.run(["nvidia-smi"], check=True)
import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch no detecta CUDA. Se detiene sin cambiar PyTorch.")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name} | VRAM {props.total_memory / 1024**3:.2f} GiB")

PERSISTENT_WORK = pathlib.Path("/kaggle/working")
SCRATCH = pathlib.Path("/kaggle/tmp/video_studio_ai")
TMP = SCRATCH / "tmp"
COMFY = SCRATCH / "ComfyUI"
OUTPUT = PERSISTENT_WORK / "video_studio_output"
WORKER_OUTPUT = PERSISTENT_WORK / "video_studio_jobs"
for folder in (SCRATCH, TMP, OUTPUT, WORKER_OUTPUT):
    folder.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = str(TMP)
os.environ["PIP_CACHE_DIR"] = str(SCRATCH / "pip-cache")
os.environ["HF_HOME"] = str(SCRATCH / "hf-home")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(SCRATCH / "hf-home" / "hub")
os.environ["COMFY_URL"] = "http://127.0.0.1:8188"
os.environ["VSA_OUTPUT_DIR"] = str(WORKER_OUTPUT)

for label, path in (("scratch", SCRATCH), ("working", PERSISTENT_WORK)):
    free = shutil.disk_usage(path).free / 1024**3
    print(label, "free GiB:", round(free, 2))
if shutil.disk_usage(PERSISTENT_WORK).free < 3 * 1024**3:
    raise RuntimeError("Se requieren al menos 3 GiB libres en /kaggle/working.")

In [ ]:
# CELDA 2 — Instalar ComfyUI y FFmpeg sin sustituir PyTorch/CUDA de Kaggle
import importlib.util

def run_cmd(args, cwd=None):
    print("$", " ".join(map(str, args)))
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)

if not (COMFY / "main.py").is_file():
    if COMFY.exists():
        raise RuntimeError(f"Existe una carpeta ComfyUI incompleta: {COMFY}. Renombra o elimina únicamente esa carpeta y vuelve a ejecutar.")
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY)])
else:
    print("Se reutiliza ComfyUI:", COMFY)

constraints = SCRATCH / "torch-constraints.txt"
protected = [f"torch=={torch.__version__}"]
for package in ("torchvision", "torchaudio"):
    if importlib.util.find_spec(package):
        module = __import__(package)
        protected.append(f"{package}=={module.__version__}")
constraints.write_text("\n".join(protected) + "\n", encoding="utf-8")
run_cmd([sys.executable, "-m", "pip", "install", "-r", str(COMFY / "requirements.txt"), "-c", str(constraints)])

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    run_cmd(["apt-get", "update", "-qq"])
    run_cmd(["apt-get", "install", "-y", "-qq", "ffmpeg"])
if not torch.cuda.is_available():
    raise RuntimeError("CUDA dejó de estar disponible después de instalar ComfyUI.")
print("ComfyUI y FFmpeg listos. PyTorch se conserva:", torch.__version__)

In [ ]:
# CELDA 3 — Descargar y verificar Wan 2.2 TI2V-5B, UMT5 y VAE
run_cmd([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
from huggingface_hub import hf_hub_download, HfApi
api = HfApi()
MODEL_SOURCES = [
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", COMFY / "models/diffusion_models"),
    ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", COMFY / "models/text_encoders"),
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/vae/wan2.2_vae.safetensors", COMFY / "models/vae"),
]
remote_sizes = {}
for repo_id, filename, _ in MODEL_SOURCES:
    info = api.model_info(repo_id, files_metadata=True)
    files = {item.rfilename: item.size for item in (info.siblings or [])}
    if filename not in files or not files[filename]:
        raise RuntimeError(f"No se pudo verificar el modelo {repo_id}/{filename}.")
    remote_sizes[(repo_id, filename)] = files[filename]

required = sum(remote_sizes.values())
free = shutil.disk_usage(SCRATCH).free
print("Modelos requeridos GiB:", round(required / 1024**3, 2), "| libres:", round(free / 1024**3, 2))
if free < required + 4 * 1024**3:
    raise RuntimeError("Espacio insuficiente para los modelos y 4 GiB de margen.")

for repo_id, filename, folder in MODEL_SOURCES:
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / pathlib.Path(filename).name
    expected = remote_sizes[(repo_id, filename)]
    if destination.is_file() and destination.stat().st_size == expected:
        print("Modelo verificado existente:", destination.name)
        continue
    if destination.exists():
        destination.unlink()
    downloaded = pathlib.Path(hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(COMFY)))
    if downloaded.resolve() != destination.resolve():
        shutil.move(str(downloaded), str(destination))
    if not destination.is_file() or destination.stat().st_size != expected:
        raise RuntimeError(f"Descarga incompleta: {destination}")
    print("Descargado y verificado:", destination.name, round(expected / 1024**3, 2), "GiB")
print("Los tres modelos están en las rutas que leerá ComfyUI.")

In [ ]:
# CELDA 4 — Iniciar una sola instancia ComfyUI y verificar el workflow Wan
import time, requests, json

BASE = os.environ["COMFY_URL"]
COMFY_LOG = SCRATCH / "comfyui.log"
required_models = [
    COMFY / "models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
    COMFY / "models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    COMFY / "models/vae/wan2.2_vae.safetensors",
]
missing = [str(path) for path in required_models if not path.is_file() or path.stat().st_size < 1_000_000]
if missing:
    raise FileNotFoundError("Faltan modelos: " + "\n".join(missing))

try:
    health = requests.get(BASE + "/system_stats", timeout=5)
    health.raise_for_status()
    print("ComfyUI ya está activo; se reutiliza.")
    server = None
except requests.RequestException:
    log_handle = open(COMFY_LOG, "a", buffering=1)
    command = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram", "--fp16-vae", "--disable-auto-launch"]
    server = subprocess.Popen(command, cwd=str(COMFY), stdout=log_handle, stderr=subprocess.STDOUT)
    ready = False
    for attempt in range(180):
        if server.poll() is not None:
            log_handle.flush()
            raise RuntimeError("ComfyUI terminó al iniciar:\n" + COMFY_LOG.read_text(errors="replace")[-12000:])
        try:
            response = requests.get(BASE + "/system_stats", timeout=5)
            if response.ok:
                ready = True
                break
        except requests.RequestException:
            pass
        time.sleep(5)
    log_handle.flush()
    if not ready:
        raise TimeoutError("ComfyUI no respondió en 15 minutos.")

object_info = requests.get(BASE + "/object_info", timeout=60)
object_info.raise_for_status()
nodes = object_info.json()
required_nodes = ["UNETLoader", "ModelSamplingSD3", "CLIPLoader", "VAELoader", "CLIPTextEncode", "Wan22ImageToVideoLatent", "LoadImage", "KSampler", "VAEDecode", "CreateVideo", "SaveVideo"]
missing_nodes = [name for name in required_nodes if name not in nodes]
if missing_nodes:
    raise RuntimeError("Faltan nodos Wan requeridos: " + ", ".join(missing_nodes))
def node_inputs(name):
    schema = nodes[name].get("input", {})
    return set(schema.get("required", {})) | set(schema.get("optional", {}))
checks = {
    "Wan22ImageToVideoLatent": {"width", "height", "length", "batch_size", "vae", "start_image"},
    "LoadImage": {"image"},
    "KSampler": {"model", "positive", "negative", "latent_image", "seed", "steps", "cfg", "sampler_name", "scheduler", "denoise"},
    "SaveVideo": {"video", "filename_prefix", "format"},
}
for name, expected in checks.items():
    missing_inputs = expected - node_inputs(name)
    if missing_inputs:
        raise RuntimeError(f"Firma incompatible del nodo {name}: {sorted(missing_inputs)}")
print("ComfyUI activo:", BASE)
print("Nodos verificados:", ", ".join(required_nodes))
print("GPU visibles:", torch.cuda.device_count(), "(la VRAM de dos tarjetas no se suma automáticamente).")

In [ ]:
# CELDA 5 — Prueba real Wan 2.2 IMAGE-TO-VIDEO (imagen inicial obligatoria)
import uuid, time, json
from PIL import Image, ImageDraw

WIDTH, HEIGHT, FRAMES, FPS = 640, 352, 49, 24
SEED = int(time.time()) % (2**32)
TEST_REFERENCE = OUTPUT / "Wan22_i2v_reference.png"
image = Image.new("RGB", (WIDTH, HEIGHT), (230, 170, 100))
draw = ImageDraw.Draw(image)
draw.rectangle((0, 0, WIDTH, 160), fill=(245, 190, 125))
draw.polygon([(0, 220), (130, 75), (270, 220), (420, 65), (640, 220), (640, 352), (0, 352)], fill=(74, 96, 78))
draw.polygon([(0, 352), (210, 245), (420, 245), (640, 352)], fill=(55, 58, 58))
draw.rounded_rectangle((235, 215, 420, 282), radius=16, fill=(175, 20, 25), outline=(25, 25, 25), width=5)
draw.polygon([(270, 215), (310, 185), (375, 185), (405, 215)], fill=(45, 60, 70))
draw.ellipse((260, 265, 315, 320), fill=(15, 15, 15), outline=(210, 210, 210), width=5)
draw.ellipse((360, 265, 415, 320), fill=(15, 15, 15), outline=(210, 210, 210), width=5)
image.save(TEST_REFERENCE)

with TEST_REFERENCE.open("rb") as stream:
    upload = requests.post(BASE + "/upload/image", files={"image": (TEST_REFERENCE.name, stream, "image/png")}, data={"type": "input", "overwrite": "true"}, timeout=180)
if not upload.ok:
    raise RuntimeError("ComfyUI rechazó la imagen de prueba: " + upload.text[:2000])
image_name = upload.json().get("name")
if not image_name:
    raise RuntimeError("ComfyUI no devolvió el nombre de la imagen.")

prompt_text = "Animate the exact supplied reference image: a red sports car accelerates on the same mountain road at sunset. Preserve the same car, wheels, road, mountains and composition. Show realistic wheel rotation, moving background parallax and a smooth tracking camera. Do not replace the car or change the scene."
negative_text = "low quality, blurry, distorted anatomy, unstable face, changing vehicle, extra wheels, flicker, artifacts, text, watermark"
workflow = {
    "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "wan2.2_ti2v_5B_fp16.safetensors", "weight_dtype": "default"}},
    "2": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["1", 0], "shift": 8}},
    "4": {"class_type": "CLIPLoader", "inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}},
    "5": {"class_type": "VAELoader", "inputs": {"vae_name": "wan2.2_vae.safetensors"}},
    "6": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt_text, "clip": ["4", 0]}},
    "7": {"class_type": "CLIPTextEncode", "inputs": {"text": negative_text, "clip": ["4", 0]}},
    "55": {"class_type": "Wan22ImageToVideoLatent", "inputs": {"width": WIDTH, "height": HEIGHT, "length": FRAMES, "batch_size": 1, "vae": ["5", 0], "start_image": ["56", 0]}},
    "56": {"class_type": "LoadImage", "inputs": {"image": image_name}},
    "3": {"class_type": "KSampler", "inputs": {"model": ["2", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["55", 0], "seed": SEED, "steps": 12, "cfg": 5, "sampler_name": "uni_pc", "scheduler": "simple", "denoise": 1}},
    "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["5", 0]}},
    "57": {"class_type": "CreateVideo", "inputs": {"images": ["8", 0], "fps": FPS}},
    "58": {"class_type": "SaveVideo", "inputs": {"video": ["57", 0], "filename_prefix": "VIDEO_STUDIO_AI/preflight", "format": "auto", "format.codec": "auto"}}
}
if workflow["55"]["inputs"]["start_image"] != ["56", 0]:
    raise RuntimeError("La prueba no está conectada a una imagen inicial.")
submitted = requests.post(BASE + "/prompt", json={"prompt": workflow, "client_id": str(uuid.uuid4())}, timeout=120)
if not submitted.ok:
    raise RuntimeError("ComfyUI rechazó el workflow I2V: " + submitted.text[:5000])
prompt_id = submitted.json().get("prompt_id")
if not prompt_id:
    raise RuntimeError("ComfyUI no devolvió prompt_id.")
deadline = time.time() + 90 * 60
record = None
while time.time() < deadline:
    response = requests.get(BASE + "/history/" + prompt_id, timeout=30)
    response.raise_for_status()
    record = response.json().get(prompt_id)
    if record:
        status = record.get("status", {})
        if status.get("status_str") == "error":
            raise RuntimeError("Wan 2.2 falló: " + json.dumps(status.get("messages", []), ensure_ascii=False)[-5000:])
        if status.get("completed") and record.get("outputs"):
            break
    time.sleep(15)
else:
    raise TimeoutError("La prueba I2V superó 90 minutos.")

items = []
for output in record.get("outputs", {}).values():
    if isinstance(output, dict):
        for key in ("videos", "gifs", "images"):
            items.extend(item for item in output.get(key, []) if isinstance(item, dict) and item.get("filename"))
mp4 = next((item for item in items if pathlib.Path(item["filename"]).suffix.lower() == ".mp4"), None)
if not mp4:
    raise RuntimeError("La prueba Wan terminó sin un MP4.")
download = requests.get(BASE + "/view", params={"filename": mp4["filename"], "subfolder": mp4.get("subfolder", ""), "type": mp4.get("type", "output")}, timeout=600)
download.raise_for_status()
TEST_VIDEO = OUTPUT / "Wan22_I2V_test.mp4"
TEST_VIDEO.write_bytes(download.content)
probe = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_name,width,height", "-of", "json", str(TEST_VIDEO)], check=True, capture_output=True, text=True)
probe_data = json.loads(probe.stdout)
actual_duration = float(probe_data["format"]["duration"])
if abs(actual_duration - FRAMES / FPS) > 0.35:
    raise RuntimeError(f"Duración inesperada: {actual_duration:.2f}s")
print("PRUEBA WAN 2.2 I2V CORRECTA:", TEST_VIDEO, "|", round(actual_duration, 2), "s |", round(TEST_VIDEO.stat().st_size / 1024**2, 2), "MiB")

In [ ]:
# CELDA 6 — Validar Piper español latino y FFmpeg antes de iniciar la cola
import urllib.request, wave, struct, json

for executable in ("ffmpeg", "ffprobe"):
    if shutil.which(executable) is None:
        raise RuntimeError(f"Falta {executable}.")

try:
    from piper import PiperVoice
except ImportError:
    run_cmd([sys.executable, "-m", "pip", "install", "--quiet", "piper-tts==1.8.0"])
    from piper import PiperVoice

VOICE_DIR = PERSISTENT_WORK / "video_studio_audio" / "voices"
VOICE_DIR.mkdir(parents=True, exist_ok=True)
VOICE_SPECS = {
    "claude": ("es_MX-claude-high.onnx", "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/es/es_MX/claude/high"),
    "ald": ("es_MX-ald-medium.onnx", "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/es/es_MX/ald/medium"),
}
for key, (model_name, base_url) in VOICE_SPECS.items():
    model_path = VOICE_DIR / model_name
    config_path = pathlib.Path(str(model_path) + ".json")
    for path, name in ((model_path, model_name), (config_path, model_name + ".json")):
        if not path.exists() or path.stat().st_size < 1000:
            urllib.request.urlretrieve(base_url + "/" + name, path)
    # Piper 1.8.0 espera valores simples ("espeak" o "text") en phoneme_type.
    # Algunas configuraciones publicadas contienen la representación Enum de Python.
    config = json.loads(config_path.read_text(encoding="utf-8"))
    phoneme_type = config.get("phoneme_type")
    if isinstance(phoneme_type, str) and phoneme_type.startswith("PhonemeType."):
        normalized = phoneme_type.split(".", 1)[1].lower()
        if normalized not in {"espeak", "text"}:
            raise ValueError(f"phoneme_type no reconocido en {config_path.name}: {phoneme_type}")
        config["phoneme_type"] = normalized
        config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"Configuración Piper normalizada: {key} | {phoneme_type} -> {normalized}")
    voice = PiperVoice.load(str(model_path), config_path=str(config_path))
    test_path = OUTPUT / f"piper_{key}_test.wav"
    with wave.open(str(test_path), "wb") as wav_file:
        voice.synthesize_wav("Hola. Esta es una prueba de voz en español latino para VIDEO STUDIO AI.", wav_file)
    with wave.open(str(test_path), "rb") as wav_file:
        frames = wav_file.readframes(wav_file.getnframes())
        sample_count = len(frames) // 2
        samples = struct.unpack("<" + "h" * sample_count, frames[:sample_count * 2])
        rms = (sum(sample * sample for sample in samples) / max(1, len(samples))) ** 0.5
        sample_rate = wav_file.getframerate()
    if len(samples) < sample_rate or rms < 80:
        raise RuntimeError(f"Piper {key} produjo audio vacío o demasiado bajo.")
    print("Voz Piper validada:", key, "| Hz:", sample_rate, "| RMS:", round(rms, 1))
print("Preflight de audio correcto. Los ambientes, efectos y música procedurales se construyen por escena con FFmpeg.")

## Iniciar el trabajador privado de Supabase

Esta celda descarga siempre los archivos del último commit de `main`, verifica sintaxis y contratos, y ejecuta la cola. No se crean ramas ni se usan claves en el código. No cierres esta sesión mientras haya trabajos en ejecución.

In [ ]:
# CELDA 7 — Leer secretos, descargar el worker de main y procesar la cola
from kaggle_secrets import UserSecretsClient
import runpy, urllib.request, hashlib, py_compile

secrets = UserSecretsClient()
SUPABASE_URL = secrets.get_secret("SUPABASE_URL").rstrip("/")
SUPABASE_SERVICE_ROLE_KEY = secrets.get_secret("SUPABASE_SERVICE_ROLE_KEY")
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
if not SUPABASE_URL.startswith("https://") or len(SUPABASE_SERVICE_ROLE_KEY) < 40:
    raise RuntimeError("Configura SUPABASE_URL y SUPABASE_SERVICE_ROLE_KEY en Kaggle Secrets.")
if len(GITHUB_TOKEN) < 20:
    raise RuntimeError("Configura GITHUB_TOKEN en Kaggle Secrets con Contents: Read-only.")

os.environ["SUPABASE_URL"] = SUPABASE_URL
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = SUPABASE_SERVICE_ROLE_KEY
os.environ["COMFY_URL"] = BASE
os.environ["VSA_OUTPUT_DIR"] = str(WORKER_OUTPUT)
os.environ["VSA_OUTPUT_BUCKET"] = "video-renders"

REPO = "sogonzalezc90-wq/video-studio-ai"
API_HEADERS = {
    "Authorization": "Bearer " + GITHUB_TOKEN,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}
commit_request = urllib.request.Request(f"https://api.github.com/repos/{REPO}/commits/main", headers=API_HEADERS)
with urllib.request.urlopen(commit_request, timeout=30) as response:
    REVISION = json.loads(response.read().decode("utf-8"))["sha"]

worker_dir = PERSISTENT_WORK / "video_studio_worker"
worker_dir.mkdir(parents=True, exist_ok=True)
if str(worker_dir) not in sys.path:
    sys.path.insert(0, str(worker_dir))

for filename in ("kaggle_supabase_worker.py", "audio_pipeline.py"):
    url = f"https://api.github.com/repos/{REPO}/contents/worker/{filename}?ref={REVISION}"
    headers = dict(API_HEADERS)
    headers["Accept"] = "application/vnd.github.raw"
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request, timeout=120) as response:
        content = response.read()
    if len(content) < 1000:
        raise RuntimeError(f"GitHub devolvió un archivo vacío/incompleto: {filename}")
    destination = worker_dir / filename
    destination.write_bytes(content)
    py_compile.compile(str(destination), doraise=True)
    print("Descarga y compilación correctas:", filename, len(content), "bytes")

worker_path = worker_dir / "kaggle_supabase_worker.py"
audio_path = worker_dir / "audio_pipeline.py"
worker_source = worker_path.read_text(encoding="utf-8")
audio_source = audio_path.read_text(encoding="utf-8")
required_worker = (
    "fetch_project_reference_media",
    "fetch_segment_master_images",
    "extract_last_frame",
    "_save_continuity_reference",
    "production_manifest",
    "continuity_reference_manifest",
)
for marker in required_worker:
    if marker not in worker_source:
        raise RuntimeError("Worker desactualizado o incompleto; falta: " + marker)
if "local_position_seconds" not in audio_source or "loudnorm" not in audio_source:
    raise RuntimeError("El módulo de audio no incluye temporización exacta y normalización.")
if not requests.get(BASE + "/system_stats", timeout=10).ok:
    raise RuntimeError("ComfyUI no responde; no se iniciará el trabajador.")
print("Commit de main:", REVISION)
print("SHA256 worker:", hashlib.sha256(worker_path.read_bytes()).hexdigest())
print("SHA256 audio:", hashlib.sha256(audio_path.read_bytes()).hexdigest())
print("Secretos leídos sin imprimir sus valores.")
print("Iniciando worker. Kaggle debe permanecer activo mientras procesa.")
runpy.run_path(str(worker_path), run_name="__main__")